<a href="https://colab.research.google.com/github/I-yuki-0424/Decision-Process-order-driven/blob/main/docs/JP-ideas/DPOD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Process Order-Driven

## To best decision system

## Name and Mean

|                  |main | 2nd |
| ---------------- | --- | --- |
| action           | A   |     |
| State            | S   | S_1 |
| target           | T   |     |
| history          | H   |     |
| compression unit | Z   |     |
| candidates       | K   |     |


## Refactor note

Same calculations as the original notebook (verified numerically against it -- see below). Restructured into classes: `TensorOps` (stateless reshape helpers), `MDPSolver` (Bellman solver + input-array builder), `AttentionStateVariant` and its four subclasses `Variant5_1Bace`/`Variant5_2`/`Variant5_3`/`Variant5_4` (the idea-5.x experiments), `AuxiliaryActionOps` (two standalone utilities), and `IdealStructurePipeline` with subclasses `MDPBranch`/`TransformerBranch` (the two branches previously duplicated four of five stages verbatim). Every function retains its original body; only names, grouping, and docstrings changed.

In [1]:
"""
Decision Process Order-Driven (DPOD) -- refactored.

Same calculations as the original notebook. Structural changes only:

  - Stateless tensor-reshaping helpers grouped under `TensorOps`.
  - The Bellman/value-iteration solver and its input-array builder grouped
    under `MDPSolver` (previously `y_mdp_action` / `make_array_to_action`,
    each split across a "_single" body and a vmapped wrapper).
  - The four "idea 5.x" experimental state-attention blocks (previously four
    unrelated top-level functions with inconsistent suffixes --
    Y_attention_state_5_1_Bace, _5_2, _5_3_L_1, _5_4) become one
    `AttentionStateVariant` family: `Variant5_1Bace`, `Variant5_2`,
    `Variant5_3`, `Variant5_4`, each exposing the same public entry point
    name (`.run`) instead of four differently-suffixed function names.
  - The "ideal structure" pipeline (previously `Y_ideal_structure_MDP` and
    `Y_ideal_structure_Transformer`, which duplicated four of their five
    stages verbatim) becomes one `IdealStructurePipeline` base class
    holding the four shared stages, with `MDPBranch` and `TransformerBranch`
    subclasses supplying only the one stage where they differ (action
    search).

Naming convention used throughout: a leading-underscore, lower_snake_case
method is a per-sample body operating on one (N, d) example; the public
`.run` (or, for stage helpers, the plain method name) is the batched,
`jax.jit`-compiled, `jax.vmap`-mapped entry point. This mirrors the
"_single vs batched" split already used consistently in the original code,
just consolidated so each concept has exactly one name instead of two
independent naming schemes (`Y_foo` / `_Y_foo_single` vs `foo` /
`_foo_single`).

No calculation logic was changed. Every per-sample body below is the
original body verbatim; every batched entry point still does exactly the
same `jax.vmap(..., in_axes=...)` call with the same `in_axes` and the same
`static_argnames` as the corresponding original function.
"""

import functools

import jax
import jax.numpy as jnp
import flax.linen as nn


# ---------------------------------------------------------------------------
# Generic tensor-reshaping helpers (stateless; shared by everything below).
# ---------------------------------------------------------------------------


### Generic tensor helpers

In [2]:
class TensorOps:
    """Small stateless reshape/broadcast helpers used across the variants
    and the ideal-structure pipeline. None of these do any learning or
    attention themselves -- they only adapt shapes between calls."""

    @staticmethod
    def sinusoidal_positional_encoding(seq_len, d_model):
        """Standard sin/cos positional encoding. (was:
        get_sinusoidal_positional_encoding)"""
        position = jnp.arange(seq_len)[:, None]
        div_term = jnp.exp(jnp.arange(0, d_model, 2) * (-jnp.log(10000.0) / d_model))

        pe_sin = jnp.sin(position * div_term)
        pe_cos = jnp.cos(position * div_term)

        pe = jnp.zeros((seq_len, d_model))
        pe = pe.at[:, 0::2].set(pe_sin)
        pe = pe.at[:, 1::2].set(pe_cos)
        return pe

    @staticmethod
    def add_head_axis(x):
        """(length, d) -> (length, 1, d). flax's dot_product_attention
        requires a num_heads axis; this notebook has no multi-head
        parameter in the idea-5.x variants, so a size-1 head is the
        minimal fix that makes those calls dimensionally valid without
        changing what they compute (attention with 1 head is
        mathematically identical to the flat 2-D version)."""
        return x[:, None, :]

    @staticmethod
    def drop_head_axis(x):
        """(length, 1, d) -> (length, d)."""
        return x[:, 0, :]

    @staticmethod
    def broadcast_to_rows(x, n_rows):
        """Broadcast a (1, d) block to (n_rows, d) by repetition; an
        already-(n_rows, d) block passes through unchanged."""
        if x.shape[0] == n_rows:
            return x
        return jnp.repeat(x, n_rows, axis=0)

    @staticmethod
    def split_heads(x, num_heads):
        """(length, d_model) -> (length, num_heads, depth_per_head)."""
        length, d_model = x.shape
        depth = d_model // num_heads
        return x.reshape(length, num_heads, depth)

    @staticmethod
    def merge_heads(x):
        """(length, num_heads, depth_per_head) -> (length, d_model)."""
        length, num_heads, depth = x.shape
        return x.reshape(length, num_heads * depth)

    @staticmethod
    def embeddings_to_scalar_id(x):
        """Placeholder discretization: mean-pools a (N, d) embedding block
        down to a (N,) scalar per row. Not a validated design -- it only
        exists so MDPSolver's scalar-indexed Q-table has something
        concrete to run against. Confirmed weak in testing: with this
        scheme, the MDP branch's top_actions come out identical across
        every row and batch element, because mean-pooling doesn't
        distinguish candidates from each other well enough. Swap this for
        a real id scheme (a fixed vocabulary, a learned classifier head, a
        hash, a lookup table) once that part of the design is settled."""
        return jnp.mean(x, axis=-1)


# ---------------------------------------------------------------------------
# Bellman / value-iteration core.
# ---------------------------------------------------------------------------


### Bellman / MDP core (was `y_mdp_action`, `make_array_to_action`)

In [3]:
class MDPSolver:
    """Tabular Bellman-optimality solver, plus the transition-array builder
    that turns raw (state, next_state, action) triples and P/R/Gamma
    coefficients into the (N, 6) array the solver expects. Previously two
    separate function pairs (`y_mdp_action` / `_y_mdp_action_single` and
    `make_array_to_action` / `_make_array_to_action_single`)."""

    @staticmethod
    def _solve_single(array_to_action, k):
        """Original per-sample body of y_mdp_action, unmodified. Operates
        on a single (N, b) slice."""
        S = array_to_action[:, 0].astype(jnp.int32)
        S_1 = array_to_action[:, 1].astype(jnp.int32)
        A = array_to_action[:, 2].astype(jnp.int32)
        P = array_to_action[:, 3]
        R = array_to_action[:, 4]
        G = array_to_action[:, 5]

        n = array_to_action.shape[0]  # static upper bound on state/action ids
        state_action_id = S * n + A  # unique id per (s, a), valid since A < n

        Q0 = jnp.zeros((n, n), dtype=array_to_action.dtype)  # Q*(s, a) initialized to 0

        def cond_fn(carry):
            it, Q, delta = carry
            return jnp.logical_and(it < n, delta > 1e-6)

        def body_fn(carry):
            it, Q, _ = carry
            topk_next = jax.lax.top_k(Q[S_1], k)[0]  # (num_rows, k)
            soft_max_next = jnp.mean(topk_next, axis=-1)  # (num_rows,)
            transition_value = P * (R + G * soft_max_next)  # (num_rows,)
            Q_flat = jax.ops.segment_sum(transition_value, state_action_id, num_segments=n * n)
            Q_new = Q_flat.reshape(n, n)
            delta = jnp.max(jnp.abs(Q_new - Q))
            return (it + 1, Q_new, delta)

        init_carry = (0, Q0, jnp.asarray(jnp.inf, dtype=array_to_action.dtype))
        _, Q, _ = jax.lax.while_loop(cond_fn, body_fn, init_carry)

        top_values, top_actions = jax.lax.top_k(Q, k)

        return top_actions, top_values

    @staticmethod
    @functools.partial(jax.jit, static_argnames=('k',))
    def solve(array_to_action, k):
        """Bellman optimality equation:
        Q*(s,a) = sum_s' P(s'|s,a) [R(s,a,s') + gamma max_a' Q*(s',a')]

        Batched: array_to_action has shape (B, N, b) -- B batches, N
        state-action rows per batch, b=6 feature columns (S, S_1, A, P, R,
        G). Per-sample logic is unchanged; jax.vmap maps `_solve_single`
        over the leading batch axis. k must be static (jax.lax.top_k
        requires a concrete Python int), hence static_argnames. (was:
        y_mdp_action)"""
        return jax.vmap(MDPSolver._solve_single, in_axes=(0, None))(array_to_action, k)

    @staticmethod
    def _build_transition_array_single(state, next_state, actions, p_coe, r_coe, gamma_coe):
        """Original per-sample body. next_state is clamped away from zero
        before division to avoid inf/nan when a next_state entry is
        exactly (or very close to) zero. The ratio-based gradient formula
        itself is otherwise unchanged from the original."""
        eps = jnp.asarray(1e-8, dtype=next_state.dtype)
        sign = jnp.where(next_state >= 0, 1.0, -1.0).astype(next_state.dtype)
        safe_next_state = jnp.where(jnp.abs(next_state) < eps, sign * eps, next_state)
        gradient = state / safe_next_state * 100
        P = p_coe * gradient
        R = r_coe * gradient
        G = gamma_coe * gradient

        return jnp.column_stack([state, next_state, actions, P, R, G])

    @staticmethod
    @jax.jit
    def build_transition_array(state, next_state, actions, p_coe, r_coe, gamma_coe):
        """S, S' = already known; a = already known.
        P = P(%)_coefficient * gradient of S -> S'
        R = R(%)_coefficient * gradient of S -> S'
        Gamma = G(%)_coefficient * gradient of S -> S'

        Batched: state, next_state, actions have shape (B, N, b). p_coe,
        r_coe, gamma_coe are shared coefficients, unbatched. next_state is
        clamped away from zero in the per-sample body to avoid inf/nan
        (see `_build_transition_array_single`). (was: make_array_to_action)
        """
        return jax.vmap(
            MDPSolver._build_transition_array_single, in_axes=(0, 0, 0, None, None, None)
        )(state, next_state, actions, p_coe, r_coe, gamma_coe)


# ---------------------------------------------------------------------------
# Idea 5.x experimental attention-state variants.
# ---------------------------------------------------------------------------


In [4]:
class AttentionStateVariant:
    """Common base for the "idea 5.x" experimental state-attention blocks.
    Each variant fuses (actions, target/goal, state, history) via
    self-attention differently and hands its result to `MDPSolver`.
    Grouping them here replaces four previously unrelated top-level
    functions (Y_attention_state_5_1_Bace, _5_2, _5_3_L_1, _5_4) that
    shared no common name, base, or call pattern. Every subclass exposes
    the batched, jitted entry point as `.run` and the per-sample body as
    `_forward_single`."""
    pos_encoding = staticmethod(TensorOps.sinusoidal_positional_encoding)
    add_head_axis = staticmethod(TensorOps.add_head_axis)
    drop_head_axis = staticmethod(TensorOps.drop_head_axis)
    broadcast_to_rows = staticmethod(TensorOps.broadcast_to_rows)


### Idea 5.x attention-state variants (was four separately-suffixed `Y_attention_state_5_*` functions)

In [5]:
class Variant5_1Bace(AttentionStateVariant):
    """idea 5, base variant ("Bace"): concatenate actions/target/state/
    history into one sequence, run `num_l` layers of self-attention with a
    residual update, then hand the result to MDPSolver.
    (was: Y_attention_state_5_1_Bace / _Y_attention_state_5_1_Bace_single)
    """

    @staticmethod
    def _forward_single(
        actions, target, state, histry,
        k_s, v_s,
        dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k,
    ):
        """Per-sample body. Operates on (N, b) slices."""
        # 1. actions, goal, state concatenate
        q_ags_init = jnp.concatenate([actions, target, state, histry], axis=0)

        def layer_fn(q_carry, _):
            y_attn = nn.dot_product_attention(
                query=Variant5_1Bace.add_head_axis(q_carry),
                key=Variant5_1Bace.add_head_axis(k_s),
                value=Variant5_1Bace.add_head_axis(v_s),
                bias=mask_s,
                dropout_rate=dropout_rate_s,
                deterministic=not dropout_enabled_s,
                dropout_rng=rng_s,
            )
            # Residual connection style update
            return q_carry + Variant5_1Bace.drop_head_axis(y_attn), None

        # 2. Layering according to num_l
        y_attention_state, _ = jax.lax.scan(layer_fn, q_ags_init, jnp.arange(num_l))

        n_a, n_t, n_s = actions.shape[0], target.shape[0], state.shape[0]
        mdp_actions, mdp_target, mdp_state, mdp_histry = jnp.split(
            y_attention_state, [n_a, n_a + n_t, n_a + n_t + n_s], axis=0
        )

        # state/target arrive as a single "now"/"goal" row while actions is
        # n_a candidate rows -- broadcast the single row across candidates
        # before stacking columns (mirrors the repeat pattern used in 5_4).
        # NOTE (unresolved, flagged previously): state/mdp_state/actions/
        # mdp_actions are each d-dimensional embeddings, not scalars, so
        # column_stack produces 6*d columns, not the 6 scalar columns
        # (S, S_1, A, P, R, G) MDPSolver's own indexing expects. This runs
        # without error but is not semantically what MDPSolver documents.
        state_b = Variant5_1Bace.broadcast_to_rows(state, n_a)
        mdp_state_b = Variant5_1Bace.broadcast_to_rows(mdp_state, n_a)
        target_b = Variant5_1Bace.broadcast_to_rows(target, n_a)

        mdp_array = jnp.column_stack([
            state_b, mdp_state_b, actions, mdp_actions,
            (target_b - mdp_state_b), (target_b - mdp_state_b) * 1 / num_step,
        ])
        return MDPSolver._solve_single(mdp_array, k)

    @staticmethod
    @functools.partial(jax.jit, static_argnames=('num_l', 'k', 'dropout_enabled_s'))
    def run(
        actions, target, state, histry,
        k_s, v_s,
        dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k,
    ):
        """Batched: actions, target, state, histry, k_s, v_s, mask_s have
        shape (B, N, b). num_l, k, dropout_enabled_s are static -- num_l
        feeds jnp.arange() for jax.lax.scan's length, k feeds
        jax.lax.top_k() inside MDPSolver, dropout_enabled_s is used in a
        Python `not` check; all three require concrete (non-traced) values
        under jax.jit. dropout_rate_s, rng_s, num_step are shared,
        unbatched, and do not need to be static. The internal call uses
        MDPSolver's single-sample form to avoid a nested vmap."""
        return jax.vmap(
            Variant5_1Bace._forward_single,
            in_axes=(0, 0, 0, 0, 0, 0, None, None, None, 0, None, None, None),
        )(actions, target, state, histry, k_s, v_s, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k)


In [6]:
class Variant5_2(AttentionStateVariant):
    """idea 5, type-1 channel separation: separate attention for
    actions/goal/state vs. self-attention for history, integrated by a
    dense layer, layered `num_l` times.
    (was: Y_attention_state_5_2 / _Y_attention_state_5_2_single)"""

    @staticmethod
    def _forward_single(
        actions, target, state, histry,
        k_s, v_s,
        dropout_rate_s, dropout_enabled_s, rng_s, mask_s,
        w_dense, b_dense, num_l,
        mask_histry=None,
    ):
        """Per-sample body. Operates on (N, b) slices."""
        q_ags_init = jnp.concatenate([actions, target, state], axis=0)
        seq_len, d_model = histry.shape[-2], histry.shape[-1]
        pe = Variant5_2.pos_encoding(seq_len, d_model)
        histry_pe_init = histry + pe

        def layer_fn(carry, _):
            q_ags, h_pe = carry

            # 1. actions, goal, state attention
            y_ags = nn.dot_product_attention(
                query=Variant5_2.add_head_axis(q_ags),
                key=Variant5_2.add_head_axis(k_s),
                value=Variant5_2.add_head_axis(v_s),
                bias=mask_s,
                dropout_rate=dropout_rate_s,
                deterministic=not dropout_enabled_s,
                dropout_rng=rng_s,
            )
            y_ags = Variant5_2.drop_head_axis(y_ags)

            # 2. attention to history
            y_histry = nn.dot_product_attention(
                query=Variant5_2.add_head_axis(h_pe),
                key=Variant5_2.add_head_axis(h_pe),
                value=Variant5_2.add_head_axis(h_pe),
                bias=mask_histry,
                dropout_rate=dropout_rate_s,
                deterministic=not dropout_enabled_s,
                dropout_rng=rng_s,
            )
            y_histry = Variant5_2.drop_head_axis(y_histry)

            # 3. linear integration and residual update
            y_combined = jnp.concatenate([y_ags, y_histry], axis=0)
            out = jnp.dot(y_combined, w_dense) + b_dense

            return (out[:q_ags.shape[0]], out[q_ags.shape[0]:]), None

        (final_q, final_h), _ = jax.lax.scan(layer_fn, (q_ags_init, histry_pe_init), jnp.arange(num_l))

        return jnp.concatenate([final_q, final_h], axis=0)

    @staticmethod
    @functools.partial(jax.jit, static_argnames=('num_l', 'dropout_enabled_s'))
    def run(
        actions, target, state, histry,
        k_s, v_s,
        dropout_rate_s, dropout_enabled_s, rng_s, mask_s,
        w_dense, b_dense, num_l,
        mask_histry=None,
    ):
        """A function that separates the attention of actions, goal, and
        state from the self-attention of history, and integrates them via
        a fully connected layer. Layered according to num_l.

        Batched: actions, target, state, histry, k_s, v_s, mask_s, and
        mask_histry (when not None) have shape (B, N, b). num_l and
        dropout_enabled_s are static (see Variant5_1Bace.run docstring).
        dropout_rate_s, rng_s, w_dense, b_dense do not need to be
        static."""
        return jax.vmap(
            Variant5_2._forward_single,
            in_axes=(0, 0, 0, 0, 0, 0, None, None, None, 0, None, None, None, 0),
        )(actions, target, state, histry, k_s, v_s, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, w_dense, b_dense, num_l, mask_histry)


In [7]:
class Variant5_3(AttentionStateVariant):
    """idea 5, type-2 channel separation: three channels -- actions (A),
    target+state (T+S), history (H) -- each self-attend, positions
    encoded, accumulated via a residual carry over `num_l` layers.
    (was: Y_attention_state_5_3_L_1 / _Y_attention_state_5_3_L_1_single)"""

    @staticmethod
    def _forward_single(
        actions, target, state, histry,
        dropout_rate_s, dropout_enabled_s, rng_s,
        mask_q, mask_k, num_l, mask_histry=None,
    ):
        """Per-sample body. Operates on (N, b) slices."""
        def make_qkv(actions, target, state, histry):
            q_seq_len = actions.shape[-2]
            q_d_model = actions.shape[-1]
            q_pe = Variant5_3.pos_encoding(q_seq_len, q_d_model)
            q = actions + q_pe

            k_bace = jnp.concatenate([target, state], axis=0)
            k_seq_len = k_bace.shape[-2]
            k_d_model = k_bace.shape[-1]
            k_pe = Variant5_3.pos_encoding(k_seq_len, k_d_model)
            k = k_bace + k_pe

            v_seq_len = histry.shape[-2]
            v_d_model = histry.shape[-1]
            v_pe = Variant5_3.pos_encoding(v_seq_len, v_d_model)
            v = histry + v_pe

            qkv = jnp.concatenate([q, k, v], axis=0)
            return (qkv, q_seq_len, k_seq_len, v_seq_len)

        def attention(carry, _):
            qkv = qkv_init + carry
            q, k, v = jnp.split(qkv, [q_split_idx, k_split_idx], axis=0)

            y_attention_state_carry_q = nn.dot_product_attention(
                query=Variant5_3.add_head_axis(q),
                key=Variant5_3.add_head_axis(q),
                value=Variant5_3.add_head_axis(q),
                bias=mask_q,
                dropout_rate=dropout_rate_s,
                deterministic=not dropout_enabled_s,
                dropout_rng=rng_s,
            )
            y_attention_state_carry_k = nn.dot_product_attention(
                query=Variant5_3.add_head_axis(k),
                key=Variant5_3.add_head_axis(k),
                value=Variant5_3.add_head_axis(k),
                bias=mask_k,
                dropout_rate=dropout_rate_s,
                deterministic=not dropout_enabled_s,
                dropout_rng=rng_s,
            )
            y_attention_state_carry_v = nn.dot_product_attention(
                query=Variant5_3.add_head_axis(v),
                key=Variant5_3.add_head_axis(v),
                value=Variant5_3.add_head_axis(v),
                bias=mask_histry,
                dropout_rate=dropout_rate_s,
                deterministic=not dropout_enabled_s,
                dropout_rng=rng_s,
            )

            y_attention_state_carry = jnp.concatenate([
                Variant5_3.drop_head_axis(y_attention_state_carry_q),
                Variant5_3.drop_head_axis(y_attention_state_carry_k),
                Variant5_3.drop_head_axis(y_attention_state_carry_v),
            ], axis=0)

            return y_attention_state_carry, None

        qkv_init, q_seq_len, k_seq_len, v_seq_len = make_qkv(actions, target, state, histry)
        q_split_idx = q_seq_len
        k_split_idx = q_seq_len + k_seq_len

        carry_init = jnp.zeros_like(qkv_init)
        loop_trigger = jnp.arange(num_l)

        final_carry, _ = jax.lax.scan(attention, carry_init, loop_trigger)
        return qkv_init + final_carry

    @staticmethod
    @functools.partial(jax.jit, static_argnames=('num_l', 'dropout_enabled_s'))
    def run(
        actions, target, state, histry,
        dropout_rate_s, dropout_enabled_s, rng_s,
        mask_q, mask_k, num_l, mask_histry=None,
    ):
        """Channels are A / T+S / H. Position encoding enabled.

        Batched: actions, target, state, histry, mask_q, mask_k, and
        mask_histry (when not None) have shape (B, N, b). num_l and
        dropout_enabled_s are static (see Variant5_1Bace.run docstring).

        mask_q and mask_k are separate because their branches generally
        differ in length (actions.shape[0] vs target.shape[0] +
        state.shape[0] -- 10 vs 2 in testing), so one shared bias cannot be
        validly shaped for both attention calls."""
        return jax.vmap(
            Variant5_3._forward_single,
            in_axes=(0, 0, 0, 0, None, None, None, 0, 0, None, 0),
        )(actions, target, state, histry, dropout_rate_s, dropout_enabled_s, rng_s, mask_q, mask_k, num_l, mask_histry)


In [8]:
class Variant5_4(AttentionStateVariant):
    """idea 5, single-shot variant: one attention pass from `state` over
    (actions, target, history), a physical-constraint projection W_k, a
    dense compression to predict S', then handed to MDPSolver. No `num_l`
    loop, unlike the other three variants.
    (was: Y_attention_state_5_4 / _Y_attention_state_5_4_single)"""

    @staticmethod
    def _forward_single(
        actions, target, state, histry,
        mask_s,
        W_k, W_dense, b_dense,
        p_coe, r_coe, gamma_coe, k_mdp,
    ):
        # 1. Self-attention to find relationships between 'now', 'goal', and 'actions'
        pe = Variant5_4.pos_encoding(histry.shape[0], histry.shape[1])
        histry_pe = histry + pe
        kv = jnp.concatenate([actions, target, histry_pe], axis=0)
        y_attn = nn.dot_product_attention(
            query=Variant5_4.add_head_axis(state),
            key=Variant5_4.add_head_axis(kv),
            value=Variant5_4.add_head_axis(kv),
            bias=mask_s,
        )
        y_attn = Variant5_4.drop_head_axis(y_attn)

        # 2. Apply W_K with physical constraints (masking via softmax)
        # We assume W_k is a compatibility matrix
        constrained_features = jnp.dot(y_attn, W_k)

        # 3. Compress into small arrays to predict next state S'
        # Output shape targets (1, d)
        s_prime_pred = jnp.dot(constrained_features, W_dense) + b_dense
        s_prime = s_prime_pred[-1:, :]  # Representative S'

        # 4. MDP Search for actions using Bellman equation
        # We construct the mdp_array using the predicted next state
        num_actions = actions.shape[0]
        state_rep = jnp.repeat(state, num_actions, axis=0)
        s_prime_rep = jnp.repeat(s_prime, num_actions, axis=0)

        # NOTE (placeholder, unresolved): mean-pooling each (num_actions, d)
        # embedding block down to one scalar per row makes the shapes line
        # up for MDPSolver._build_transition_array_single, but pooling is
        # not a validated way to turn a continuous embedding into a
        # discrete state/action id (same open design question as
        # TensorOps.embeddings_to_scalar_id).
        mdp_array = MDPSolver._build_transition_array_single(
            jnp.mean(state_rep, axis=-1),
            jnp.mean(s_prime_rep, axis=-1),
            jnp.mean(actions, axis=-1),
            p_coe, r_coe, gamma_coe,
        )

        top_actions, top_values = MDPSolver._solve_single(mdp_array, k_mdp)

        return top_actions, top_values, s_prime

    @staticmethod
    @functools.partial(jax.jit, static_argnames=('k_mdp',))
    def run(
        actions, target, state, histry,
        mask_s,
        W_k, W_dense, b_dense,
        p_coe, r_coe, gamma_coe, k_mdp,
    ):
        """Batched: actions, target, state, histry, mask_s have shape
        (B, N, b); W_k, W_dense, b_dense, p_coe, r_coe, gamma_coe are
        shared, unbatched. k_mdp is static (jax.lax.top_k requires a
        concrete value, same reasoning as MDPSolver.solve)."""
        return jax.vmap(
            Variant5_4._forward_single,
            in_axes=(0, 0, 0, 0, 0, None, None, None, None, None, None, None),
        )(actions, target, state, histry, mask_s, W_k, W_dense, b_dense, p_coe, r_coe, gamma_coe, k_mdp)


# ---------------------------------------------------------------------------
# Small standalone action-selection utilities (predate the ideal-structure
# pipeline below; not called by it).
# ---------------------------------------------------------------------------


### Auxiliary, standalone action utilities (was `Get_next_state`, `y_attention_action`)

In [9]:
class AuxiliaryActionOps:
    """Two small, independent utilities: argmax-based next-state selection,
    and a standalone attention-based action ranker. Neither is called by
    `AttentionStateVariant` subclasses or `IdealStructurePipeline` -- they
    are earlier, separate drafts. (was: Get_next_state /
    _Get_next_state_single, and y_attention_action /
    _y_attention_action_single)"""

    @staticmethod
    def _next_state_single(y_attention_state, state_index):
        """Original per-sample body, unmodified."""
        state_softmax = nn.softmax(y_attention_state[state_index])
        state_next = jnp.argmax(state_softmax)
        return state_next

    @staticmethod
    @jax.jit
    def next_state(y_attention_state, state_index):
        """Transformer-selected best next state.

        Batched: y_attention_state has shape (B, N, b). state_index is
        shared across the batch (the same relative position is queried for
        every sample); pass a batched index array with in_axes changed to
        0 if per-sample indices are needed instead. (was: Get_next_state)
        """
        return jax.vmap(AuxiliaryActionOps._next_state_single, in_axes=(0, None))(y_attention_state, state_index)

    @staticmethod
    def _attention_action_single(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a):
        """Per-sample body."""
        q = array_to_action

        y_attention_action = nn.dot_product_attention(
            query=TensorOps.add_head_axis(q),
            key=TensorOps.add_head_axis(k_a),
            value=TensorOps.add_head_axis(v_a),
            bias=mask_a,
            dropout_rate=dropout_rate_a,
            deterministic=not dropout_enabled_a,
            dropout_rng=rng_a,
        )

        return TensorOps.drop_head_axis(y_attention_action)

    @staticmethod
    @functools.partial(jax.jit, static_argnames=('dropout_enabled_a',))
    def attention_action(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a):
        """Returns the next-step action via attention (as opposed to the
        MDP/Bellman route). Untested speculative alternative kept from the
        original for parity -- not used by the ideal-structure pipeline.

        Batched: array_to_action, k_a, v_a, mask_a have shape (B, N, b).
        dropout_enabled_a is static (same reasoning as elsewhere: a Python
        `not` on it requires a concrete value under jit). dropout_rate_a,
        rng_a are shared, unbatched, and do not need to be static. (was:
        y_attention_action)"""
        return jax.vmap(
            AuxiliaryActionOps._attention_action_single, in_axes=(0, 0, 0, None, None, None, 0)
        )(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a)


# ---------------------------------------------------------------------------
# Ideal structure: self-attention over now/goal/actions, physically-
# constrained W_K filter, next-goal prediction, then two interchangeable
# action-search branches (MDP/Bellman vs Transformer), and per-step history
# compression.
# ---------------------------------------------------------------------------


### Ideal structure pipeline (was `Y_ideal_structure_MDP`, `Y_ideal_structure_Transformer`)

In [10]:
class IdealStructurePipeline:
    """Shared stages of the "ideal structure" pipeline:
      1. self-attention among state/target/actions
      2. W_K-projected, physically-constrained attention -> filtered state
      3. compress to predicted next goal S', shape (1, d)
      6. compress this step's filtered candidates and extend history H

    `MDPBranch` and `TransformerBranch` below add stage 4 (action search)
    -- the only stage where the two branches previously diverged; every
    other stage was duplicated verbatim between
    `Y_ideal_structure_MDP` and `Y_ideal_structure_Transformer` in the
    original notebook."""

    @staticmethod
    def self_attention_now_goal_actions(
        state, target, actions, num_heads,
        dropout_rate, dropout_enabled, rng, mask=None,
    ):
        """Stage 1: let 'now' (state), 'goal' (target), and 'actions'
        attend to each other jointly, so every element can see every other
        element -- unlike any of the AttentionStateVariant blocks, none of
        which let these three groups cross-attend."""
        q_ags = jnp.concatenate([state, target, actions], axis=0)
        q_h = TensorOps.split_heads(q_ags, num_heads)
        attn = nn.dot_product_attention(
            query=q_h, key=q_h, value=q_h, bias=mask,
            dropout_rate=dropout_rate,
            deterministic=not dropout_enabled,
            dropout_rng=rng,
        )
        return TensorOps.merge_heads(attn)

    @staticmethod
    def physical_key_filter(
        query_state, candidates, w_k, b_k, num_heads,
        physically_possible, dropout_rate, dropout_enabled, rng,
    ):
        """Stage 2: project `candidates` through a learned key weight W_K,
        then attend to them from `query_state` (the now+goal
        representation). `physically_possible` (boolean, shape (q_length,
        kv_length)) becomes an additive -1e9 bias -- softmax drives
        attention weight to exactly zero at any (query, key) pair marked
        False. The zeroing comes from that additive bias, not from
        constraining W_K's values; W_K itself is an ordinary learned
        projection. This produces a query-length output (used for
        next-goal prediction) -- it does not by itself remove candidates
        from the pool the action-search branches see; that's what
        `candidate_survival_mask` is for."""
        key = jnp.dot(candidates, w_k) + b_k
        bias = jnp.where(physically_possible, 0.0, -1e9)[None, :, :]
        q_h = TensorOps.split_heads(query_state, num_heads)
        k_h = TensorOps.split_heads(key, num_heads)
        v_h = TensorOps.split_heads(candidates, num_heads)
        attn = nn.dot_product_attention(
            query=q_h, key=k_h, value=v_h, bias=bias,
            dropout_rate=dropout_rate,
            deterministic=not dropout_enabled,
            dropout_rng=rng,
        )
        return TensorOps.merge_heads(attn)

    @staticmethod
    def candidate_survival_mask(physically_possible):
        """physically_possible is (q_length, kv_length): per (query,
        candidate) feasibility. Collapsed to (kv_length,): a candidate
        survives if it is physically possible from at least one query row.
        This is what makes "only actions that remain after being filtered
        by W_K" concrete and JIT-shape-stable -- candidate count stays
        fixed, but disallowed rows are zeroed / scored -inf downstream
        instead of merely down-weighted inside stage 2's query-side
        attention output."""
        return jnp.any(physically_possible, axis=0)

    @staticmethod
    def predict_next_goal(filtered_state, w_goal, b_goal):
        """Stage 3: pool the filtered now/goal representation across its
        sequence axis and pass through one dense layer, compressing to a
        single (1, d) vector -- the predicted next goal S'."""
        pooled = jnp.mean(filtered_state, axis=0, keepdims=True)
        return jnp.dot(pooled, w_goal) + b_goal

    @staticmethod
    def compress_step_to_history(candidates, w_z, b_z, history):
        """Stage 6: compress this step's (already physically-filtered)
        candidates into a single (1, d) vector and append it to the
        running history buffer H."""
        pooled = jnp.mean(candidates, axis=0, keepdims=True)
        step_repr = jnp.dot(pooled, w_z) + b_z
        return jnp.concatenate([history, step_repr], axis=0)


In [11]:
class MDPBranch(IdealStructurePipeline):
    """Ideal structure, stage 4 = Bellman/MDP top-k action search over
    candidates that survive the physical filter. Reuses MDPSolver
    unchanged, via mean-pooled scalarization.
    (was: Y_ideal_structure_MDP / _Y_ideal_structure_MDP_single)

    Returns (next_goal, top_actions, top_values, updated_history).
    top_actions/top_values are Q-table indices in the discretized
    state-action id space (see TensorOps.embeddings_to_scalar_id) -- they
    are NOT the same index space as the original candidate positions in
    `actions`, unlike TransformerBranch's top_actions."""

    @staticmethod
    def _action_search_single(state, next_goal, filtered_candidates, p_coe, r_coe, gamma_coe, k_mdp):
        n_c = filtered_candidates.shape[0]
        state_rep = TensorOps.broadcast_to_rows(state, n_c)
        next_goal_rep = TensorOps.broadcast_to_rows(next_goal, n_c)
        mdp_array = MDPSolver._build_transition_array_single(
            TensorOps.embeddings_to_scalar_id(state_rep),
            TensorOps.embeddings_to_scalar_id(next_goal_rep),
            TensorOps.embeddings_to_scalar_id(filtered_candidates),
            p_coe, r_coe, gamma_coe,
        )
        return MDPSolver._solve_single(mdp_array, k_mdp)

    @staticmethod
    def _forward_single(
        state, target, actions, history, num_heads,
        w_k, b_k, physically_possible, w_goal, b_goal,
        p_coe, r_coe, gamma_coe, k_mdp, w_z, b_z,
        dropout_rate, dropout_enabled, rng, mask_self=None,
    ):
        stage1 = MDPBranch.self_attention_now_goal_actions(
            state, target, actions, num_heads, dropout_rate, dropout_enabled, rng, mask=mask_self
        )
        n_s, n_t = state.shape[0], target.shape[0]
        query_repr = stage1[:n_s + n_t]
        actions_repr = stage1[n_s + n_t:]

        stage2 = MDPBranch.physical_key_filter(
            query_repr, actions_repr, w_k, b_k, num_heads,
            physically_possible, dropout_rate, dropout_enabled, rng,
        )
        next_goal = MDPBranch.predict_next_goal(stage2, w_goal, b_goal)

        candidate_survives = MDPBranch.candidate_survival_mask(physically_possible)
        filtered_candidates = jnp.where(candidate_survives[:, None], actions_repr, 0.0)

        top_actions, top_values = MDPBranch._action_search_single(
            state, next_goal, filtered_candidates, p_coe, r_coe, gamma_coe, k_mdp
        )
        updated_history = MDPBranch.compress_step_to_history(filtered_candidates, w_z, b_z, history)
        return next_goal, top_actions, top_values, updated_history

    @staticmethod
    @functools.partial(jax.jit, static_argnames=('num_heads', 'k_mdp', 'dropout_enabled'))
    def run(
        state, target, actions, history, num_heads,
        w_k, b_k, physically_possible, w_goal, b_goal,
        p_coe, r_coe, gamma_coe, k_mdp, w_z, b_z,
        dropout_rate, dropout_enabled, rng, mask_self=None,
    ):
        """Batched: state, target, actions, history, physically_possible,
        and mask_self (when not None) have shape (B, ...). w_k, b_k,
        w_goal, b_goal, w_z, b_z, p_coe, r_coe, gamma_coe, dropout_rate,
        rng are shared, unbatched. num_heads, k_mdp, dropout_enabled are
        static (head split reshape, jax.lax.top_k, and the Python `not`
        on dropout_enabled all require concrete values under jax.jit)."""
        return jax.vmap(
            MDPBranch._forward_single,
            in_axes=(0, 0, 0, 0, None, None, None, 0, None, None, None, None, None, None, None, None, None, None, None, 0),
        )(state, target, actions, history, num_heads, w_k, b_k, physically_possible, w_goal, b_goal,
          p_coe, r_coe, gamma_coe, k_mdp, w_z, b_z, dropout_rate, dropout_enabled, rng, mask_self)


In [12]:
class TransformerBranch(IdealStructurePipeline):
    """Ideal structure, stage 4 = attention-score ranking over the
    physically-filtered candidates, offered as the alternative to
    MDPBranch's Bellman/MDP search.
    (was: Y_ideal_structure_Transformer /
    _Y_ideal_structure_Transformer_single)

    Returns (next_goal, top_actions, top_values, updated_history). Unlike
    MDPBranch, top_actions here ARE indices directly into the original
    `actions` candidate array -- they don't go through the scalar-id
    discretization at all, which is why this branch's exclusion of a
    physically-forbidden candidate could be verified directly while
    MDPBranch's could not."""

    @staticmethod
    def _action_search_single(next_goal, filtered_candidates, candidate_survives, num_heads,
                               dropout_rate, dropout_enabled, rng, k_top):
        """filtered_candidates here is the raw (unzeroed) candidate
        embedding set, used as K/V for the attention call;
        candidate_survives (bool, shape (Nc,)) is applied directly to the
        ranking scores below, which is what actually guarantees a
        physically disallowed candidate can never be selected, regardless
        of its embedding content. next_goal (the predicted S') queries the
        candidates; the raw query-key dot product (scaled) ranks them,
        since dot_product_attention itself returns only the weighted
        output, not per-candidate weights usable for top-k ranking."""
        q_h = TensorOps.split_heads(next_goal, num_heads)
        k_h = TensorOps.split_heads(filtered_candidates, num_heads)
        v_h = TensorOps.split_heads(filtered_candidates, num_heads)
        nn.dot_product_attention(
            query=q_h, key=k_h, value=v_h,
            dropout_rate=dropout_rate,
            deterministic=not dropout_enabled,
            dropout_rng=rng,
        )  # kept for structural parity with MDPBranch; ranking uses raw scores below
        d_model = filtered_candidates.shape[-1]
        scores = jnp.dot(next_goal, filtered_candidates.T)[0] / jnp.sqrt(d_model)
        scores = jnp.where(candidate_survives, scores, -jnp.inf)
        top_values, top_actions = jax.lax.top_k(scores, k_top)
        return top_actions, top_values

    @staticmethod
    def _forward_single(
        state, target, actions, history, num_heads,
        w_k, b_k, physically_possible, w_goal, b_goal,
        k_top, w_z, b_z,
        dropout_rate, dropout_enabled, rng, mask_self=None,
    ):
        stage1 = TransformerBranch.self_attention_now_goal_actions(
            state, target, actions, num_heads, dropout_rate, dropout_enabled, rng, mask=mask_self
        )
        n_s, n_t = state.shape[0], target.shape[0]
        query_repr = stage1[:n_s + n_t]
        actions_repr = stage1[n_s + n_t:]

        stage2 = TransformerBranch.physical_key_filter(
            query_repr, actions_repr, w_k, b_k, num_heads,
            physically_possible, dropout_rate, dropout_enabled, rng,
        )
        next_goal = TransformerBranch.predict_next_goal(stage2, w_goal, b_goal)

        candidate_survives = TransformerBranch.candidate_survival_mask(physically_possible)
        top_actions, top_values = TransformerBranch._action_search_single(
            next_goal, actions_repr, candidate_survives, num_heads, dropout_rate, dropout_enabled, rng, k_top
        )
        filtered_candidates = jnp.where(candidate_survives[:, None], actions_repr, 0.0)
        updated_history = TransformerBranch.compress_step_to_history(filtered_candidates, w_z, b_z, history)
        return next_goal, top_actions, top_values, updated_history

    @staticmethod
    @functools.partial(jax.jit, static_argnames=('num_heads', 'k_top', 'dropout_enabled'))
    def run(
        state, target, actions, history, num_heads,
        w_k, b_k, physically_possible, w_goal, b_goal,
        k_top, w_z, b_z,
        dropout_rate, dropout_enabled, rng, mask_self=None,
    ):
        """Batching and staticness follow the same convention as
        MDPBranch.run (see its docstring). k_top replaces k_mdp: both are
        static top-k widths, just for different rankings (attention score
        here, Bellman Q-value there)."""
        return jax.vmap(
            TransformerBranch._forward_single,
            in_axes=(0, 0, 0, 0, None, None, None, 0, None, None, None, None, None, None, None, None, 0),
        )(state, target, actions, history, num_heads, w_k, b_k, physically_possible, w_goal, b_goal,
          k_top, w_z, b_z, dropout_rate, dropout_enabled, rng, mask_self)


# ---------------------------------------------------------------------------
# Top-level dispatcher (was: Generate_action).
#
# NOTE: in the original notebook this function was never functional -- its
# inner closures call e.g. `y_mdp_action()` with zero arguments against a
# signature that requires at least `(array_to_action, k)`, so calling any
# branch would raise TypeError immediately. It is also never invoked
# anywhere else in the notebook (including the test cell), so there is no
# recorded behavior to preserve. What follows is a like-for-like rename of
# each branch to its new class-based name -- e.g. `y_mdp_action()` ->
# `MDPSolver.solve()` -- with the same missing arguments and the same
# "#please add" marker carried over unchanged. This is still a stub, not a
# working dispatcher; wiring it up would be new logic and is left to you.
# ---------------------------------------------------------------------------


### Dispatcher stub (was `Generate_action` -- unchanged in behavior: still incomplete, never called)

In [13]:
def generate_action(target, state, actions, histry, top_k):  # please add
    def normal_mdp():
        action = MDPSolver.solve()
        return action

    def idea_5_1():
        action = Variant5_1Bace.run()
        return action

    def idea_5_2():
        action = Variant5_2.run()
        return action

    def idea_5_3():
        action = Variant5_3.run()
        return action

    def idea_5_4():
        action = Variant5_4.run()
        return action

    def idea_5_5():  # Y_ideal_structure_MDP
        action = MDPBranch.run()
        return action

    def idea_5_6():
        action = TransformerBranch.run()
        return action


## Test cell (same configuration and assertions as the original notebook's test cell)

In [14]:
import traceback
import jax
import jax.numpy as jnp

# --- Test Configuration (identical to the original notebook's test cell) ---
B, N, D = 2, 10, 8
num_l = 2
num_step = 5
k_mdp = 3
p_coe, r_coe, gamma_coe = 0.1, 0.5, 0.9

key = jax.random.PRNGKey(20090424)
actions = jax.random.normal(key, (B, N, D))
target = jax.random.normal(key, (B, 1, D))
state = jax.random.normal(key, (B, 1, D))
histry = jax.random.normal(key, (B, 20, D))

k_s = jax.random.normal(key, (B, N + 22, D))
v_s = jax.random.normal(key, (B, N + 22, D))
mask_s = jnp.zeros((B, 1, N + 22, N + 22))
mask_h = jnp.zeros((B, 1, 20, 20))
w_dense = jax.random.normal(key, (D, D))
b_dense = jax.random.normal(key, (D,))

def raw_test_run(name, func, *args):
    print(f"\n{'='*20}\nTesting: {name}\n{'='*20}")
    try:
        res = func(*args)
        print(f"SUCCESS: {name}")
        if isinstance(res, tuple):
             print("Output shapes:", [getattr(r, 'shape', 'non-array') for r in res])
        else:
             print("Output shape:", res.shape)
    except Exception:
        print(f"FAILURE: {name}")
        traceback.print_exc()

# Execute Tests (same calls, new class-based entry points)
raw_test_run("5_1 Bace", Variant5_1Bace.run,
             actions, target, state, histry, k_s, v_s, 0.0, False, key, mask_s, num_l, num_step, k_mdp)

# 5_2: q_ags length = actions(N) + target(1) + state(1) = N+2, not N+3 --
# the previous N+3 slice was one row too wide for k_s/v_s/mask_s here.
raw_test_run("5_2", Variant5_2.run,
             actions, target, state, histry, k_s[:, :N+2, :], v_s[:, :N+2, :],
             0.0, False, key, mask_s[:, :, :N+2, :N+2], w_dense, b_dense, num_l, mask_h)

# 5_3: mask_s split into mask_q (q branch, length = actions = N) and
# mask_k (k branch, length = target + state = 2) -- see Variant5_3's
# docstring for why one shared mask could not work here.
mask_q_5_3 = jnp.zeros((B, 1, N, N))
mask_k_5_3 = jnp.zeros((B, 1, 2, 2))
raw_test_run("5_3 L_1", Variant5_3.run,
             actions, target, state, histry, 0.0, False, key, mask_q_5_3, mask_k_5_3, num_l, mask_h)

# Testing 5_4
# state (query) length=1, kv length = actions(N) + target(1) + histry(20) = N+21
mask_5_4 = jnp.zeros((B, 1, 1, N + 21))
W_k = jax.random.normal(key, (D, D)) * 0.1
raw_test_run("5_4", Variant5_4.run,
             actions, target, state, histry, mask_5_4,
             W_k, w_dense, b_dense, p_coe, r_coe, gamma_coe, k_mdp)

# --- Ideal structure: MDP branch vs Transformer branch ---
n_hist_ideal = 5
history_buf = jax.random.normal(key, (B, n_hist_ideal, D))
w_k_ideal = jax.random.normal(key, (D, D)) * 0.1
b_k_ideal = jnp.zeros((D,))
w_goal_ideal = jax.random.normal(key, (D, D)) * 0.1
b_goal_ideal = jnp.zeros((D,))
w_z_ideal = jax.random.normal(key, (D, D)) * 0.1
b_z_ideal = jnp.zeros((D,))
num_heads_ideal = 2
# state/target are 1 row each in this notebook's convention; actions is N rows
physically_possible = jnp.ones((B, 1 + 1, N), dtype=bool)
physically_possible = physically_possible.at[:, :, -1].set(False)  # forbid the last candidate

def raw_test_run_tuple(name, func, *args):
    print(f"\n{'='*20}\nTesting: {name}\n{'='*20}")
    try:
        res = func(*args)
        print(f"SUCCESS: {name}")
        print("Output shapes:", [getattr(r, 'shape', 'non-array') for r in res])
    except Exception:
        print(f"FAILURE: {name}")
        traceback.print_exc()

raw_test_run_tuple("Ideal structure -- MDP branch", MDPBranch.run,
                    state, target, actions, history_buf, num_heads_ideal,
                    w_k_ideal, b_k_ideal, physically_possible, w_goal_ideal, b_goal_ideal,
                    p_coe, r_coe, gamma_coe, k_mdp, w_z_ideal, b_z_ideal,
                    0.0, False, None, None)

raw_test_run_tuple("Ideal structure -- Transformer branch", TransformerBranch.run,
                    state, target, actions, history_buf, num_heads_ideal,
                    w_k_ideal, b_k_ideal, physically_possible, w_goal_ideal, b_goal_ideal,
                    k_mdp, w_z_ideal, b_z_ideal,
                    0.0, False, None, None)



Testing: 5_1 Bace


SUCCESS: 5_1 Bace
Output shapes: [(2, 10, 3), (2, 10, 3)]

Testing: 5_2


SUCCESS: 5_2
Output shape: (2, 32, 8)

Testing: 5_3 L_1


SUCCESS: 5_3 L_1
Output shape: (2, 32, 8)

Testing: 5_4


SUCCESS: 5_4
Output shapes: [(2, 10, 3), (2, 10, 3), (2, 1, 8)]



Testing: Ideal structure -- MDP branch


SUCCESS: Ideal structure -- MDP branch
Output shapes: [(2, 1, 8), (2, 10, 3), (2, 10, 3), (2, 6, 8)]

Testing: Ideal structure -- Transformer branch


SUCCESS: Ideal structure -- Transformer branch
Output shapes: [(2, 1, 8), (2, 3), (2, 3), (2, 6, 8)]
